# <font color=#c51b8a>OpsiGen / RhoMax Applications</font>
### Training and prediction workflows for structure-aware opsin lambda max models

This notebook is a practical, executable companion to the OpsiGen documentation.

**Scope of this notebook**

- Use the existing OpsiGen/RhoMax model to predict lambda max for a new opsin when a matching PDB structure is available.
- Prepare data and graph features for training a new model.
- Train a PyTorch Geometric RhoMax-style model from a config file.
- Package a trained model with its normalization artifacts and metadata.

### Important Note
OpsiGen/RhoMax is structure-aware. It does not predict directly from raw sequence alone. The practical input is an amino-acid sequence plus a matching protein structure, usually generated with AlphaFold2 or ColabFold.


# <font color=#c994c7>Step 0: OpsiGen Setup</font>
### Define paths, import packages, and check the local repository state

Edit `opsigen_root_path` if this notebook is moved outside the `docs/` directory. When the notebook is stored in `docs/`, `Path('..').resolve()` should point to the OpsiGen repository root.


In [ ]:
# All necessary packages to import for data processing and workflow orchestration.
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import numpy as np
import pandas as pd


In [ ]:
# Edit this path if the notebook is not run from OpsiGen/docs.
opsigen_root_path = Path('..').resolve()

predict_dir = opsigen_root_path / 'predict'
pipeline_dir = opsigen_root_path / 'pipeline_auto'
feature_maker_dir = opsigen_root_path / 'feature_maker'
excel_dir = opsigen_root_path / 'excel'
docs_dir = opsigen_root_path / 'docs'

print('OpsiGen root:', opsigen_root_path)
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())


In [ ]:
# Repository contract checks. These should all exist in the current checkout.
required_paths = {
    'phenotype table': excel_dir / 'data.xlsx',
    'reference alignment': excel_dir / 'sequences.fas',
    'wavelength labels': excel_dir / 'wavelength.dat',
    'split directory': excel_dir / 'splits',
    'prediction config': predict_dir / 'conf_all',
    'legacy model': predict_dir / 'model_pickel',
    'legacy means': predict_dir / 'means.npy',
    'legacy stds': predict_dir / 'stds.npy',
    'single-query pipeline config': pipeline_dir / 'config.json',
    'feature maker binary': feature_maker_dir / 'interface2grid',
}

for label, path in required_paths.items():
    print(f'{label:28s}', 'OK' if path.exists() else 'MISSING', path)


In [ ]:
# Optional dependency checks. PyTorch/PyG imports can fail if the environment has not been built yet.
def check_import(module_name):
    try:
        module = __import__(module_name)
        version = getattr(module, '__version__', 'version unknown')
        print(f'{module_name:20s} OK {version}')
        return module
    except Exception as exc:
        print(f'{module_name:20s} MISSING/ERROR: {exc}')
        return None

_ = check_import('torch')
_ = check_import('torch_geometric')
_ = check_import('Bio')

mafft_path = shutil.which('mafft')
print('mafft'.ljust(20), mafft_path or 'MISSING FROM PATH')


# <font color=#c994c7>Step 1a: Inspect The Training Data Table</font>
### Confirm the phenotype table has the schema expected by the current `PDBDataset`

The current training loader reads `excel/data.xlsx`. Required operational columns are `Name`, `Wildtype`, `Sequence`, and `lmax`. The historical table also includes `Method` and 24 binding-pocket audit columns.


In [ ]:
phenotype_file = excel_dir / 'data.xlsx'
df = pd.read_excel(phenotype_file)

print('shape:', df.shape)
print('columns:', list(df.columns))
print('unique wildtypes:', df['Wildtype'].nunique())
print('lmax range:', df['lmax'].min(), 'to', df['lmax'].max())

df.head()


In [ ]:
# Basic curation checks before model training.
required_columns = ['Name', 'Wildtype', 'Sequence', 'lmax']
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

checks = {
    'unique Name values': df['Name'].is_unique,
    'no missing Wildtype': df['Wildtype'].notna().all(),
    'no missing Sequence': df['Sequence'].notna().all(),
    'numeric lmax': pd.api.types.is_numeric_dtype(df['lmax']),
    'no missing lmax': df['lmax'].notna().all(),
}

for label, passed in checks.items():
    print(f'{label:24s}', 'PASS' if passed else 'FAIL')

sequence_lengths = df['Sequence'].astype(str).str.len()
print('sequence length min/median/max:', sequence_lengths.min(), sequence_lengths.median(), sequence_lengths.max())


# <font color=#c994c7>Step 1b: Inspect Wildtype-Aware Train/Test Splits</font>
### Avoid leakage by splitting wildtypes, not individual mutant rows

Most mutant records are closely related to their parent wildtype. OpsiGen/RhoMax evaluation should keep all variants of a wildtype in the same split.


In [ ]:
splits_dir = excel_dir / 'splits'

def read_split(name):
    return [line.strip() for line in (splits_dir / name).read_text().splitlines() if line.strip()]

split_summary = []
for split_id in range(5):
    train_wt = read_split(f'train{split_id}')
    test_wt = read_split(f'test{split_id}')
    overlap = sorted(set(train_wt).intersection(test_wt))
    train_rows = df[df['Wildtype'].isin(train_wt)].shape[0]
    test_rows = df[df['Wildtype'].isin(test_wt)].shape[0]
    split_summary.append({
        'split': split_id,
        'train_wildtypes': len(train_wt),
        'test_wildtypes': len(test_wt),
        'overlap_count': len(overlap),
        'train_rows': train_rows,
        'test_rows': test_rows,
    })

pd.DataFrame(split_summary)


In [ ]:
# Visual sanity check of lambda max distributions by split.
# This mirrors the paper-level concern that split performance depends on train/test spectral coverage.
import matplotlib.pyplot as plt

split_id_to_plot = 0
train_wt = read_split(f'train{split_id_to_plot}')
test_wt = read_split(f'test{split_id_to_plot}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df[df['Wildtype'].isin(train_wt)]['lmax'], bins=20, alpha=0.65, label='train')
ax.hist(df[df['Wildtype'].isin(test_wt)]['lmax'], bins=20, alpha=0.65, label='test')
ax.set_xlabel('lambda max (nm)')
ax.set_ylabel('count')
ax.set_title(f'Split {split_id_to_plot} lambda max distribution')
ax.legend()
plt.show()


# <font color=#c994c7>Step 2: Generate Or Validate Structure Graph Features</font>
### Convert sequence + PDB inputs into graph tensors for model training or prediction

For every trainable row, OpsiGen expects two files named by row index:

```text
cutted_parts{i}.npz
cutted_parts{i}_dists.npy
```

The current sample single-query pipeline writes the same pattern with `i = 0` under `pipeline_auto/features/` and `pipeline_auto/dists/`.


In [ ]:
# Inspect the committed sample feature tensors.
sample_features = pipeline_dir / 'features' / 'cutted_parts0.npz'
sample_dists = pipeline_dir / 'dists' / 'cutted_parts0_dists.npy'

if sample_features.exists() and sample_dists.exists():
    features = np.load(sample_features)
    dists = np.load(sample_dists)
    print('features:', features.shape, features.dtype, 'min/max:', np.nanmin(features), np.nanmax(features))
    print('dists:', dists.shape, dists.dtype, 'min/max:', np.nanmin(dists), np.nanmax(dists))
else:
    print('Sample feature tensors are missing. Run the feature-generation cells below after setting inputs.')


In [ ]:
# Configure a single-query feature-generation run.
# The current pipeline expects exactly one FASTA record and one matching PDB.
query_fasta_path = pipeline_dir / 'sample_fasta.fasta'
query_pdb_path = pipeline_dir / 'sample_pdb.pdb'

pipeline_config_path = pipeline_dir / 'config.json'
pipeline_config = json.loads(pipeline_config_path.read_text())
pipeline_config


In [ ]:
# Optional: copy external query files into the legacy single-query pipeline filenames.
# Set COPY_INPUTS_TO_PIPELINE_AUTO = True after editing the source paths.
COPY_INPUTS_TO_PIPELINE_AUTO = False
external_query_fasta = Path('/path/to/query.fasta')
external_query_pdb = Path('/path/to/query.pdb')

if COPY_INPUTS_TO_PIPELINE_AUTO:
    shutil.copyfile(external_query_fasta, query_fasta_path)
    shutil.copyfile(external_query_pdb, query_pdb_path)
    print('Copied query FASTA and PDB into pipeline_auto/.')
else:
    print('Not copying inputs. Edit paths above and set COPY_INPUTS_TO_PIPELINE_AUTO = True when ready.')


In [ ]:
# Optional long-running/platform-dependent step: generate graph features for the query FASTA/PDB.
# This requires MAFFT and a working feature_maker/interface2grid binary.
RUN_FEATURE_GENERATION = False

if RUN_FEATURE_GENERATION:
    # Remove only the standard single-query outputs so stale files do not mask errors.
    standard_outputs = [
        pipeline_dir / 'cutted_parts' / 'cutted_parts0.pdb',
        pipeline_dir / 'features' / 'cutted_parts0.npz',
        pipeline_dir / 'dists' / 'cutted_parts0_dists.npy',
        pipeline_dir / 'mafft_alignments' / f"{pipeline_config.get('mafft_id', 'mafft_id_0')}.fasta",
    ]
    for folder_name in ['dists', 'features', 'cutted_parts', 'mafft_alignments']:
        (pipeline_dir / folder_name).mkdir(exist_ok=True)
    for path in standard_outputs:
        if path.exists():
            path.unlink()

    cmd = [sys.executable, 'main.py', 'config.json']
    print('Running:', ' '.join(cmd), 'in', pipeline_dir)
    subprocess.run(cmd, cwd=pipeline_dir, check=True)
else:
    print('Set RUN_FEATURE_GENERATION = True to generate cut PDB, features, and distance matrices.')


# <font color=#c994c7>Step 3: Train A RhoMax Graph Model</font>
## This is the long section

The current training entry point is `predict/train.py`. It reads a JSON config, constructs `PDBDataset`, computes training-set normalization statistics, trains a selected model class from `predict/models.py`, and saves the best checkpoint by test-set mean absolute error.

Use split-specific configs for evaluation. Use an all-wildtype config only when intentionally fitting a final production model after evaluation.


In [ ]:
# Load the base training config.
base_config_path = predict_dir / 'conf_all'
base_config = json.loads(base_config_path.read_text())
base_config


In [ ]:
# Define a notebook run directory and create a split-specific config.
# For strict evaluation, change split_id and repeat across split0..split4.
model_version_label = 'notebook_rhomax_demo'
split_id = 0
run_dir = predict_dir / 'runs' / model_version_label / f'split{split_id}'
run_dir.mkdir(parents=True, exist_ok=True)

def rel_to_predict(path):
    return os.path.relpath(Path(path).resolve(), predict_dir)

training_config = dict(base_config)
training_config.update({
    'excel_path': rel_to_predict(excel_dir / 'data.xlsx'),
    'graph_features_path': '../db_features/',
    'graph_dists_path': '../db_dists/',
    'train_wildtypes_list': rel_to_predict(splits_dir / f'train{split_id}'),
    'test_wildtypes_list': rel_to_predict(splits_dir / f'test{split_id}'),
    'epochs': 1000,
    'seed': 123,
    'wandb': False,
    'checkpoint_dir': str(run_dir / 'checkpoints'),
    'means_path': str(run_dir / 'means.npy'),
    'stds_path': str(run_dir / 'stds.npy'),
    'pickle_file': 'opsigen_rhomax_model.pkl',
})

training_config_path = run_dir / 'conf.json'
print(training_config_path)
training_config


In [ ]:
# Write the training config to disk.
# This is safe and small; rerun after editing paths or hyperparameters.
training_config_path.write_text(json.dumps(training_config, indent=2) + '\n')
print('Wrote:', training_config_path)


In [ ]:
# Audit whether graph tensors exist for rows included by the selected split files.
# A full training run needs features and dists for the selected phenotype rows.
graph_features_path = (predict_dir / training_config['graph_features_path']).resolve()
graph_dists_path = (predict_dir / training_config['graph_dists_path']).resolve()
train_wt = set(read_split(f'train{split_id}'))
test_wt = set(read_split(f'test{split_id}'))
selected_indices = df.index[df['Wildtype'].isin(train_wt.union(test_wt))].tolist()

missing = []
for idx in selected_indices:
    f = graph_features_path / f'cutted_parts{idx}.npz'
    d = graph_dists_path / f'cutted_parts{idx}_dists.npy'
    if not f.exists() or not d.exists():
        missing.append((idx, f.exists(), d.exists()))

print('selected rows:', len(selected_indices))
print('missing graph rows:', len(missing))
print('first missing rows:', missing[:10])
print('feature directory:', graph_features_path)
print('distance directory:', graph_dists_path)


In [ ]:
# Optional long-running step: train the model.
# Leave RUN_TRAINING = False until graph_features_path and graph_dists_path are complete.
RUN_TRAINING = False

if RUN_TRAINING:
    cmd = [sys.executable, 'train.py', str(training_config_path)]
    print('Running:', ' '.join(cmd), 'in', predict_dir)
    subprocess.run(cmd, cwd=predict_dir, check=True)
else:
    print('Set RUN_TRAINING = True after confirming all graph tensors exist.')


# <font color=#c994c7>Step 4: Evaluate And Inspect Training Artifacts</font>
### Keep metrics, normalization arrays, configs, and checkpoints together

A reusable model is not just a `.pkl` file. It also needs the exact normalization arrays and training config used to create it.


In [ ]:
# Find checkpoints created by the training run.
checkpoint_dir = Path(training_config['checkpoint_dir'])
if not checkpoint_dir.is_absolute():
    checkpoint_dir = predict_dir / checkpoint_dir

checkpoints = sorted(checkpoint_dir.glob('*.pkl'))
print('checkpoint_dir:', checkpoint_dir)
print('checkpoints:', len(checkpoints))
for path in checkpoints[-5:]:
    print(path.name)


In [ ]:
# Inspect normalization artifacts.
means_path = Path(training_config['means_path'])
stds_path = Path(training_config['stds_path'])
if not means_path.is_absolute():
    means_path = predict_dir / means_path
if not stds_path.is_absolute():
    stds_path = predict_dir / stds_path

if means_path.exists() and stds_path.exists():
    means = np.load(means_path)
    stds = np.load(stds_path)
    print('means:', means.shape, means.dtype)
    print('stds:', stds.shape, stds.dtype)
    print('non-constant columns:', int((stds != 0).sum()))
else:
    print('Normalization arrays not found yet. They are written during training.')


In [ ]:
# Recommended run manifest for reproducibility.
manifest = {
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    'opsigen_root_path': str(opsigen_root_path),
    'training_config': str(training_config_path),
    'python': sys.version,
    'platform': platform.platform(),
    'model_version_label': model_version_label,
    'split_id': split_id,
}

manifest_path = run_dir / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n')
print('Wrote:', manifest_path)
manifest


# <font color=#c994c7>Step 5: Predict With The Existing Legacy Model</font>
### Use precomputed or newly generated graph tensors to predict lambda max

The direct prediction script is `predict/calculate_one_rhodopsin.py`. It takes:

1. A config file.
2. A pickled PyTorch model.
3. An output text file.
4. A feature matrix file.
5. A distance matrix file.

The committed legacy model is `predict/model_pickel`, with normalization arrays `predict/means.npy` and `predict/stds.npy` referenced by `predict/conf_all`.


In [ ]:
legacy_config_path = predict_dir / 'conf_all'
legacy_model_path = predict_dir / 'model_pickel'
legacy_output_path = pipeline_dir / 'result.txt'
legacy_feature_path = pipeline_dir / 'features' / 'cutted_parts0.npz'
legacy_dist_path = pipeline_dir / 'dists' / 'cutted_parts0_dists.npy'

for label, path in {
    'legacy_config': legacy_config_path,
    'legacy_model': legacy_model_path,
    'output_path_parent': legacy_output_path.parent,
    'feature_path': legacy_feature_path,
    'dist_path': legacy_dist_path,
}.items():
    print(f'{label:20s}', 'OK' if path.exists() else 'MISSING', path)


In [ ]:
# Optional prediction step using existing sample graph tensors.
RUN_EXISTING_MODEL_PREDICTION = False

if RUN_EXISTING_MODEL_PREDICTION:
    cmd = [
        sys.executable,
        'calculate_one_rhodopsin.py',
        str(legacy_config_path),
        str(legacy_model_path),
        str(legacy_output_path),
        str(legacy_feature_path),
        str(legacy_dist_path),
    ]
    print('Running:', ' '.join(cmd), 'in', predict_dir)
    subprocess.run(cmd, cwd=predict_dir, check=True)
    print(legacy_output_path.read_text())
else:
    print('Set RUN_EXISTING_MODEL_PREDICTION = True to run the legacy model on the current sample tensors.')


# <font color=#c994c7>Step 6: Query A Newly Trained Model To Predict NEW Sequences</font>
### This mirrors the final query section in the VPOD workflow notebook

After training, use the best checkpoint plus its matching `means.npy` and `stds.npy`. Do not mix normalization arrays across runs.


In [ ]:
# Select a trained checkpoint. If no checkpoint exists yet, this cell reports that training is still needed.
trained_checkpoints = sorted(checkpoint_dir.glob('*.pkl'))
trained_model_path = trained_checkpoints[-1] if trained_checkpoints else None
print('trained_model_path:', trained_model_path)


In [ ]:
# Create a prediction config that points to the newly trained normalization arrays.
# This keeps the model checkpoint and normalization statistics paired.
if trained_model_path is not None and means_path.exists() and stds_path.exists():
    trained_prediction_config = dict(training_config)
    trained_prediction_config['means_path'] = str(means_path)
    trained_prediction_config['stds_path'] = str(stds_path)
    trained_prediction_config_path = run_dir / 'prediction_conf.json'
    trained_prediction_config_path.write_text(json.dumps(trained_prediction_config, indent=2) + '\n')
    print('Wrote:', trained_prediction_config_path)
else:
    trained_prediction_config_path = None
    print('No trained model/normalization pair found yet.')


In [ ]:
# Optional prediction with a newly trained checkpoint on the current pipeline_auto graph tensors.
RUN_TRAINED_MODEL_PREDICTION = False
trained_output_path = run_dir / 'trained_model_prediction.txt'

if RUN_TRAINED_MODEL_PREDICTION:
    if trained_prediction_config_path is None or trained_model_path is None:
        raise RuntimeError('Train a model first, then rerun the previous cells.')
    cmd = [
        sys.executable,
        'calculate_one_rhodopsin.py',
        str(trained_prediction_config_path),
        str(trained_model_path),
        str(trained_output_path),
        str(legacy_feature_path),
        str(legacy_dist_path),
    ]
    print('Running:', ' '.join(cmd), 'in', predict_dir)
    subprocess.run(cmd, cwd=predict_dir, check=True)
    print(trained_output_path.read_text())
else:
    print('Set RUN_TRAINED_MODEL_PREDICTION = True after training and feature generation.')


# <font color=#c994c7>Step 7: Package A Model For Reuse</font>
### Bundle the checkpoint, normalization arrays, config, and reproducibility metadata

Use this section when a model is ready to be used by someone else or by a later prediction workflow. A model bundle should be self-describing.


In [ ]:
# Optional model packaging step.
PACKAGE_MODEL = False
bundle_dir = opsigen_root_path / 'models' / model_version_label

if PACKAGE_MODEL:
    if trained_model_path is None or not means_path.exists() or not stds_path.exists():
        raise RuntimeError('Cannot package until a trained model, means.npy, and stds.npy exist.')

    bundle_dir.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(trained_model_path, bundle_dir / 'model.pkl')
    shutil.copyfile(means_path, bundle_dir / 'means.npy')
    shutil.copyfile(stds_path, bundle_dir / 'stds.npy')
    shutil.copyfile(training_config_path, bundle_dir / 'train_config.json')
    shutil.copyfile(manifest_path, bundle_dir / 'manifest.json')

    feature_contract = """# Feature Contract

- Input graph feature file: cutted_parts{i}.npz
- Input distance file: cutted_parts{i}_dists.npy
- Raw feature shape: (N, 36)
- N: number of atoms in the 24 selected binding-pocket residues
- Prediction target: lambda max in nm
"""
    (bundle_dir / 'feature_contract.md').write_text(feature_contract)
    print('Packaged model bundle:', bundle_dir)
else:
    print('Set PACKAGE_MODEL = True after selecting a final checkpoint.')


# <font color=#c994c7>Step 8: Reproducibility Checklist</font>

Before publishing or comparing a model, make sure the following are archived:

- Phenotype table version.
- Split files.
- Structure-generation protocol and software version.
- Graph feature-generation code version.
- Training config JSON.
- Python, PyTorch, and PyTorch Geometric versions.
- Random seed.
- Model checkpoint.
- `means.npy` and `stds.npy` from the same training run.
- Test-set predictions and metrics.
- Notes on applicability domain, especially taxonomic scope and lambda max range.

The most common reproducibility failures are mixing model checkpoints with the wrong normalization arrays, splitting mutants away from their wildtype, and silently skipping rows because graph files are missing.


# <font color=#c994c7>Troubleshooting Notes</font>

**`mafft: command not found`**

Install MAFFT with conda or your system package manager, then rerun the setup checks.

**`feature_maker/interface2grid` cannot execute**

The committed binary is Linux-only. Use Linux, Docker, WSL2, or rebuild the feature maker for your platform.

**Training reports no usable graphs**

Check that `graph_features_path` and `graph_dists_path` point to complete row-indexed tensors and that the split files contain wildtype names present in the phenotype table.

**Feature dimension mismatch**

Check that raw feature arrays are `(N, 36)`, that `indexes_to_keep` is correct, and that `number_features` matches the number of non-constant columns after applying the training-set `stds.npy` mask.

**Prediction is outside the expected range**

Inspect the query structure confidence, pocket-residue mapping, taxonomic distance from the training set, and whether the query lambda max is likely outside the training distribution.
